# QuackWeave usage patterns for read-only Ladybug graphs

**Design target.** Use standalone GQL or Cypher when the question is about a graph. Use SQL/PGQ when one SQL statement needs both graph matches and relational data. In both cases the query defines its result once. Projected types come from the graph schema and the expressions.

| User task | Public query form | Reference |
|---|---|---|
| Query a selected graph in GQL | `MATCH … RETURN …` | ISO/IEC 39075:2024 |
| Query a selected graph in Cypher | `MATCH … RETURN …` | openCypher 9 specification and TCK |
| Join graph matches to a table or Parquet-backed view | `SELECT … FROM GRAPH_TABLE (graph MATCH … COLUMNS (…)) … JOIN …` | ISO/IEC 9075-16:2023, SQL/PGQ |
| Reuse the exact text of a standalone query in an analytical workflow | Execute the graph statement in its dialect, then expose its typed result as a relation for SQL | Client integration, described under P11 |

**Implemented scope.** The native extension runs the read-only patterns below on DuckDB 2.0.1 and Ladybug 0.22.3. Select a graph for a standalone statement, name the graph in a SQL/PGQ table expression, or transfer a standalone result to a SQL client using Arrow. The coverage table links each entry point to its fixture tests. The explicit-schema `qw_query` table function remains available. The appendix records its earlier baseline. These fixture results establish support for the features listed here. Full language conformance would require broader testing.

[ISO GQL](https://www.iso.org/standard/76120.html) · [openCypher specification and tests](https://opencypher.org/resources/) · [ISO SQL/PGQ](https://www.iso.org/standard/79473.html)

## Graph selection and data used in the examples

Provision the Ladybug file before using it in DuckDB. Load the native extension in a DuckDB instance that allows unsigned extensions, replace the two paths below, and register the existing graph. This implementation supports one configured alias per instance.

```sql
LOAD '/absolute/path/to/quack_weave.duckdb_extension';
SET GLOBAL qw_ladybug_config = '{"alias":"air_routes","path":"/absolute/path/to/air_routes.lbdb"}';
SESSION SET GRAPH air_routes;
```

The graph selection and dialect belong to the DuckDB connection. Standalone statements default to GQL. To run the same text as Cypher, execute `SET SESSION qw_graph_dialect = 'cypher'`. To switch back, use `'gql'`. A pooled connection that was already open when another connection loaded the extension can activate the grammar with `CALL add_grammar_extension('quack_weave')`. In SQL/PGQ, name the registered graph directly inside `GRAPH_TABLE`. The table expression resolves its graph without a session selection. [Session acceptance cases](test/stabilization/sqllogic/graph_entry.test) · [DuckGQL session selection](https://duckgql.com/docs/guides/querying.html)

### Fixture contract

The graph has nodes labeled `airport` with a unique string property `code`. Each directed `route` has a fixture-unique integer property `rid` and a nullable 64-bit integer property `dist`. Fixture values are illustrative.

| rid | Origin | Destination | dist |
|---:|---|---|---:|
| 101 | ATL | BOS | 946 |
| 102 | ATL | BOS | 946 |
| 103 | ATL | LAX | 1946 |
| 104 | ATL | PDX | 2172 |
| 105 | ATL | SEA | NULL |

Also create an isolated airport `ORD`. The DuckDB relation `airport_directory(code, city)` has four entries with unique codes, namely `BOS → Boston`, `LAX → Los Angeles`, `SEA → Seattle`, and `ORD → Chicago`. It can be a table or a view over Parquet. PDX deliberately has no directory row. The two BOS edges exercise duplicate preservation, SEA exercises nullable properties, and ORD exercises empty graph results.

## P1 · Standalone graph query

With the graph and dialect selected, submit the graph statement directly. Run the air-routes query below unchanged in each dialect.

```sql
MATCH (origin:airport)-[route:route]->(destination:airport)
WHERE origin.code = 'ATL'
RETURN origin.code AS origin,
       destination.code AS destination,
       route.dist AS miles
LIMIT 10;
```

The three result columns, in projection order, are `origin` (string), `destination` (string), and `miles` (nullable 64-bit integer on this fixture). Expect five rows in any order, including two identical `ATL, BOS, 946` rows and one `ATL, SEA, NULL` row. Replacing ATL with ORD gives zero rows with the same metadata. A limit without ordering does not identify which ten rows to keep on a larger graph.

**Result semantics.** Resolve column names, aliases, expressions, nulls, duplicates, `DISTINCT`, ordering and paging according to the selected dialect. Test names without explicit aliases under that dialect’s naming rules. A supported `RETURN route` returns a relationship value with its identity. A node or path projection likewise retains its graph semantics. If the implementation cannot represent a value or evaluate a feature faithfully, report the unsupported feature explicitly. The four scalar types in the current bridge do not define the language’s result model. [ISO GQL](https://www.iso.org/standard/76120.html) · [openCypher 9 resources and TCK](https://opencypher.org/resources/)

## P2–P3 · One SQL statement for graph traversal and analytics

Find the two longest routes from ATL whose destinations have directory entries. `MATCH` binds graph elements, `COLUMNS` exposes their properties to SQL, and the outer query joins, sorts and limits the resulting rows. This is SQL/PGQ syntax. [SQL/PGQ](https://www.iso.org/standard/79473.html) · [Graph table examples](https://duckpgq.org/documentation/sql_pgq/)

```sql
WITH routes AS (
    SELECT g.*
    FROM GRAPH_TABLE (
        air_routes
        MATCH (o:airport)-[e:route]->(d:airport)
        WHERE o.code = 'ATL'
        COLUMNS (e.rid AS route_id, o.code AS origin,
                 d.code AS destination, e.dist AS miles)
    ) AS g
)
SELECT r.origin, r.destination, a.city, r.miles
FROM routes AS r
JOIN airport_directory AS a ON a.code = r.destination
ORDER BY r.miles DESC NULLS LAST, r.route_id
FETCH FIRST 2 ROWS ONLY;
```

| origin | destination | city | miles |
|---|---|---|---:|
| ATL | LAX | Los Angeles | 1946 |
| ATL | BOS | Boston | 946 |

PDX has the longest route and drops out at the inner join. Ranking graph rows and taking two **before** the join would select PDX and LAX, leaving only LAX after the join. `route_id` breaks the tie between the BOS edges. A SQL CTE names the intermediate result. Its physical scan count is a host planning decision.

## P4–P7 · Reusing a graph-derived SQL relation

The examples below use a single view over the named graph (P7). Its graph projection is bound from the catalog and its rows are read when the view is queried.

```sql
CREATE VIEW atl_routes AS
SELECT g.*
FROM GRAPH_TABLE (
    air_routes
    MATCH (o:airport)-[e:route]->(d:airport)
    WHERE o.code = 'ATL'
    COLUMNS (e.rid AS route_id, d.code AS destination, e.dist AS miles)
) AS g;
```

**P4 · Join two graph-derived relations.** To find parallel routes to the same destination, self-join the view. The inequality gives each pair once. The fixture yields the single pair `BOS, 101, 102`.

```sql
SELECT l.destination, l.route_id AS first_route, r.route_id AS second_route
FROM atl_routes AS l
JOIN atl_routes AS r
  ON l.destination = r.destination AND l.route_id < r.route_id;
```

**P5 · Use a subquery for existence or a scalar.** Return each matching directory entry once, even when several routes end there. For one scalar per directory entry, a correlated `SELECT COUNT(*)` also handles the zero-match case.

```sql
SELECT a.code
FROM airport_directory AS a
WHERE EXISTS (SELECT 1 FROM atl_routes AS r WHERE r.destination = a.code)
ORDER BY a.code;
```

Expected codes are BOS, LAX and SEA. ORD has no matching route. `IN` on the one-column destination projection is also appropriate for these non-null airport codes.

**P6 · Combine and aggregate.** `UNION ALL` retains the five route rows and the four directory rows. `UNION` on the same single-column projections yields the five distinct codes BOS, LAX, ORD, PDX and SEA.

```sql
SELECT destination AS code FROM atl_routes
UNION ALL
SELECT code FROM airport_directory;
```

For aggregation, `SELECT COUNT(*) AS routes, COUNT(miles) AS known_distances, COUNT(DISTINCT destination) AS destinations FROM atl_routes` returns `5, 4, 4`. The two BOS routes both contribute to the route count. Only the four non-null distances contribute to `COUNT(miles)`.

## P8 · Parameters in the language that consumes them

An outer SQL predicate can use the host client’s prepared-statement binding. This DuckDB example queries the SQL/PGQ view defined under P7.

```sql
PREPARE count_routes AS
SELECT COUNT(*) AS routes
FROM atl_routes
WHERE destination = $1;

EXECUTE count_routes('BOS');
EXECUTE count_routes('ORD');
```

The executions return 2 and 0. The positional placeholder belongs to DuckDB. In a standalone GQL or Cypher predicate, bind a named value using the same statement text for each execution.

```sql
PREPARE flights AS
MATCH (o:airport)-[e:route]->(d:airport)
WHERE o.code = $origin
RETURN d.code AS destination;

EXECUTE flights(origin := 'ATL');
EXECUTE flights(origin := 'ORD');
```

The first execution returns five rows, the second zero. The DuckDB ADBC driver binds by position, so Python can execute this same graph statement with `cursor.execute(statement, ("ATL",))`. Both paths bind typed values through Ladybug. Graph and property names are identifiers. The SQLLogic suite also binds SQL parameters directly in `GRAPH_TABLE` predicates and projected expressions. [Standalone parameters](test/stabilization/sqllogic/graph_parameters.test) · [SQL/PGQ parameters](test/stabilization/sqllogic/graph_table.test)

**Acceptance.** Run the same prepared SQL twice with different codes. For each supported graph-language parameter facility, execute the same graph text with ATL and ORD and assert five rows and zero rows respectively. Inputs remain data, including values containing quotes.

## P9 · Read-only Ladybug integration

Use an existing, populated Ladybug graph. A standalone read and a hybrid join both open the graph for read-only access. Creating a DuckDB view, joining a directory table, or materializing query results in DuckDB changes only host state. Graph creation, CSV import, mutation and deletion are outside this Ladybug usage target.

**Acceptance.** Establish the fixture contents before execution. Attempt a graph update in a standalone write-capable dialect and assert rejection by the read-only integration. Exercise the supported read statements, repeat the fixture observations, and confirm graph contents and edge multiplicities. The SQL/PGQ matching table expression is a read. [Architecture](quack-weave-architecture.md)

## P10–P12 · Client integration and query planning

**P10 · Look up host data in SQL.** Project the graph properties needed for matching, then use the directory table in a SQL join as in P3. If an airport lacks directory data, a `LEFT JOIN` can keep the route. This read-only target requires no DuckDB functions or SQL statements embedded inside graph-language text.

**P11 · Reuse an exact GQL or Cypher result.** Some applications already own a standalone `MATCH … RETURN` query. Execute it in the selected dialect, retain its column metadata, then expose its returned rows through the client’s relation-registration API. The executable example uses DuckDB’s ADBC driver. `source` and `sink` below are cursors on two independent DuckDB databases. `graph_statement` is the P1 text.

```python
source.execute(graph_statement)
sink.adbc_ingest("route_result", source.fetch_arrow_table())
```

In this example the receiving database owns a materialized copy. Its SQL query reads the values that the graph returned. [Client example](test/public_client/run.py) · [Consumer assertions](test/public_client/results.test)

Query the resulting `route_result(origin, destination, miles)` with ordinary SQL.

```sql
SELECT r.destination, a.city, r.miles
FROM route_result AS r
JOIN airport_directory AS a ON a.code = r.destination
ORDER BY r.destination, r.miles;
```

For the P1 result the join yields four rows, including both BOS rows. A graph-side limit is already applied before registration. SQL callers specify their own ordering. For repeatable SQL-only use, a view such as P7 avoids the client handoff. Bind conversion must preserve values and nullability for every supported type.

**P12 · Explain and execute.** In DuckDB a plain `EXPLAIN` of a hybrid statement must plan its projected types without opening a Ladybug execution cursor or executing a traversal. Make graph catalog metadata available to the binder. `EXPLAIN ANALYZE` executes the query and may report an open or read error. Distinguish a catalog binding failure from a scan failure. Exact plan operator names and profiler counters are host diagnostics.

**Binding lifetime.** A bound scan must detect a changed configuration generation before opening. Fresh binding of a named graph resolves the graph and its current schema again. Reuse of a saved query across independent executions carries no promise that an administrator has kept the alias pointed at the same file.

## Join keys, missing data and ordering

Use the projected business key `destination` to join to `airport_directory.code`. The fixture makes both keys strings. Internal graph element identity is useful for distinguishing parallel edges within one graph. A cross-source join needs a key whose meaning both datasets share.

| Operation on the fixture | Expected result |
|---|---|
| Inner join `atl_routes` to the directory | Four rows: BOS twice, LAX once and SEA once |
| Left join from `atl_routes` | Five rows, including PDX with a null city |
| `COUNT(miles)` over the graph view | Four known distances. SEA still joins to Seattle because the join uses the airport code. |
| Equality join with a null key in an additional SQL fixture row | No match under SQL `=` semantics |
| Inner join after adding a second BOS directory row | Six rows. The two BOS routes each match both directory rows. |
| Outer ordering by distance with ties | Specify `NULLS LAST` and a stable tie-breaker such as `route_id` to test an exact sequence |

Unmatched keys are valid data. When testing a business rule that requires every graph key to have a directory row, assert it separately. The earlier JEV foreign-key examples modeled an optional fixture integrity rule (`sol_28a17ec62ad84a47` and `sol_7eb621ea7ab14594`). The PDX fixture intentionally exercises a valid join with an unmatched key.

## SQLLogic and fixture acceptance

Run the **public statement text** on a real DuckDB instance with the read-only Ladybug provider. Run P1 separately with the GQL and Cypher entry points. Run hybrid statements through the SQL entry point. Attach a case ID, entry point, fixture version and expected result or error to each test. An unavailable language feature stays visibly pending or explicitly unsupported. A passing bridge query cannot satisfy a public-syntax case.

| Public cases | Fixture assertion |
|---|---|
| P1 graph selection and projection | Select `air_routes` for one GQL connection. Check that selection is connection-local. Run the supplied ATL statement unchanged in each supported dialect. Check the five rows, duplicate BOS tuple, null SEA distance, projected names, column order and types. Repeat with ORD and check the empty result’s metadata. |
| P1 language semantics | Exercise explicit and implicit result names, `DISTINCT`, nullable properties and deterministic ordered paging. Project a node, edge and path independently. For a supported relationship result, assert the two BOS edges have distinct identities and the same endpoint codes. Pair unsupported graph-value cases with explicit diagnostics. Use the openCypher TCK as the Cypher reference. |
| P2–P3 CTE, join and paging | Execute the complete SQL/PGQ query above and compare the two ordered rows. In a separate case rank the graph rows first, take two, then join and assert LAX alone. Use a table and a Parquet-backed directory view with the same input. |
| P4–P7 SQL composition | Check the parallel-edge self-join, semi-join, both set operations, counts, inner and left joins. Query the named view twice. Test an unmatched and a duplicated directory key. An already-bound scan must detect an alias change, while a freshly bound query must validate its current metadata. |
| P8 bound values | Execute the prepared host predicate for BOS and ORD. For a supported graph parameter, bind ATL, ORD and a quoted value without changing statement text. |
| P9 read-only access | Assert the write rejection and compare fixture contents before and after reads and the rejection. |
| P10–P11 result reuse | Check the four-row directory join, including both BOS rows. Where the host client registers graph results, assert column metadata and row count at registration, then check the downstream SQL result. |
| P12 planning | Plan the query against a registered schema with an unreadable backing file. Assert no graph scan opens during plain `EXPLAIN`. Force execution and check the provider open error. |

Use SQLLogic `rowsort` when the language leaves order unspecified. Preserve duplicate output lines. For ordered paging, include a tie-breaker in the query and compare rows in sequence. Assert names and logical types separately from values, using DuckDB metadata (or the driver metadata for standalone results). A zero-row value comparison alone cannot establish the schema.

Run `make native-stabilization-test` for the pinned DuckDB SQLLogic runner and its Ladybug observer. Run `make public-client-test` for the ADBC result handoff. The first target provisions an isolated graph and checks its digest before and after test execution. The second provisions a separate graph and uses SQLLogic to inspect the receiving database. [Air-routes fixture](test/stabilization/fixtures/air_routes.cypher) · [SQLLogic suites](test/stabilization/sqllogic) · [Client assertions](test/public_client/results.test)

## Implemented coverage and boundaries

| Pattern | Behavior exercised on the pinned engines | Executable evidence |
|---|---|---|
| P1 selection and projection | Bare GQL and Cypher return five ATL rows and a typed empty ORD result. Selection and language are connection-local. The binder resolves supported projections from the registered catalog. | [Statement entry](test/stabilization/sqllogic/graph_entry.test) · [Projection metadata](test/stabilization/sqllogic/graph_metadata.test) |
| P1 graph values | Nodes and relationships expose identity, labels and properties as typed DuckDB structs. Named paths expose ordered node and relationship lists. Paths between differently labeled nodes preserve optional properties when the common field types agree. Parallel edges retain distinct identities. | [Graph-value cases](test/stabilization/sqllogic/graph_values.test) |
| P2–P3 graph matches and SQL | Named, directed fixed patterns in `GRAPH_TABLE` work inside CTEs and joins. The order of paging and the directory join determines whether BOS survives. | [Table expressions](test/stabilization/sqllogic/graph_table.test) · [Hybrid queries](test/stabilization/sqllogic/hybrid_public.test) |
| P4–P7 relational composition | Self-joins, semi-joins, set operations, views, duplicate directory keys and missing values are checked with fixture rows. Parquet supplies a second directory source. | [Hybrid SQLLogic](test/stabilization/sqllogic/hybrid_public.test) |
| P8 typed parameters | SQL prepared values work in outer SQL and inside `GRAPH_TABLE`. Standalone GQL and Cypher bind named scalar values. Inputs containing quotes remain data. | [Graph parameters](test/stabilization/sqllogic/graph_parameters.test) · [Table expressions](test/stabilization/sqllogic/graph_table.test) |
| P9 read-only enforcement | Admission rejects graph mutations. Fixture digests and provider observations check the read-only boundary. | [Hybrid SQLLogic](test/stabilization/sqllogic/hybrid_public.test) · [Stabilization harness](test/stabilization/run_sqllogictest.py) |
| P10–P11 SQL and client reuse | SQL joins preserve the requested row multiplicities. Two independent DuckDB databases exchange the exact graph result through Arrow. The consumer verifies the empty-result schema and relationship identity. | [ADBC example](test/public_client/run.py) · [Consumer SQLLogic](test/public_client/results.test) |
| P12 planning | Plain `EXPLAIN` resolves types from registered metadata. Execution against a damaged file reports a provider failure. | [Hybrid SQLLogic](test/stabilization/sqllogic/hybrid_public.test) |

The fixture uses a single configured, read-only graph. SQL/PGQ supports explicitly labeled directed patterns and scalar SQL expressions in the tested positions. Quantified SQL/PGQ paths and aggregates inside `COLUMNS` produce errors. Standalone GQL covers a single labeled node or a fixed directed edge. Graph-value properties support BOOLEAN, BIGINT, DOUBLE and VARCHAR. Modeled graph features and their client representation are tested against Ladybug and DuckDB. The ISO standards and openCypher specify the respective languages. [ISO GQL](https://www.iso.org/standard/76120.html) · [openCypher resources](https://opencypher.org/resources/) · [ISO SQL/PGQ](https://www.iso.org/standard/79473.html)

### Per-task checks

Each change has a fixture-backed SQLLogic test. Pre-implementation JEV checks describe the expected invariant. Post-implementation checks use the fixture or host results. The SQLLogic runner exercises the engine and package behavior directly.

| Task | Pre-implementation JEV / Z3 | Post-implementation JEV / Z3 |
|---|---|---|
| Air-routes fixture | `sol_0fbaa26e569e4b2c` | `sol_5d55f3031956487f` |
| Graph result metadata | `sol_be6a9e905b3147ac` | `sol_4edf3dd6fcf94719` |
| Session selection and native statements | `sol_8f6619a4612f45ad` | `sol_582f9f26947a4c5c` |
| SQL/PGQ table expressions | `sol_c7bac6a2136d4519` | `sol_74e0647f413b4229` |
| Hybrid joins and paging | `sol_59c7ac5fbd4e4267` | `sol_5a3a50caa541442a` |
| Prepared graph parameters | `sol_c1ccc76166454543` | `sol_83dc995f8306480f` |
| Graph-valued projections | `sol_8d97b91ec9ae49fa` | `sol_d3080364c15b42e7` |
| ADBC result handoff | `sol_70eaef821b524432` | `sol_da98fee053834863` for the directory join, `sol_d404b0273b5b4182` for distinct edge IDs |

The final stabilization report records 138 cases and 268 case/profile combinations. The separate ADBC consumer passed 11 SQLLogic blocks plus its engine-version check.

### Run the public examples

The next cell loads the release extension in the pinned DuckDB CLI and runs both standalone dialects against a fresh Ladybug file. It then runs the hybrid join through SQL/PGQ. Run the kernel from the repository, or set `QW_REPO` to the checkout. Set `DUCKDB`, `LBUG`, or `QW_EXTENSION` to override the default paths.

In [2]:
import os
import sys
import tempfile
from pathlib import Path

root = Path(os.environ.get("QW_REPO", Path.cwd())).resolve()
sys.path.insert(0, str(root / "test"))
from cypher_acceptance import duckdb, ladybug, provision, read_rows, sql_literal

cli = Path(os.environ.get("DUCKDB", root / "build/deps/duckdb-v2.0.1/duckdb"))
lbug = Path(os.environ.get("LBUG", root / "build/deps/ladybug-v0.22.3/lbug"))
package = Path(os.environ.get("QW_EXTENSION", root / "build/native/release/extension/quack_weave/quack_weave.duckdb_extension"))
fixture = root / "test/stabilization/fixtures/air_routes.cypher"
graph_statement = """MATCH (origin:airport)-[route:route]->(destination:airport)
WHERE origin.code = 'ATL'
RETURN origin.code AS origin, destination.code AS destination, route.dist AS miles
LIMIT 10;"""
hybrid = """SELECT a.city, g.miles
FROM GRAPH_TABLE (
    air_routes MATCH (o:airport)-[e:route]->(d:airport)
    WHERE o.code = 'ATL' COLUMNS (d.code AS destination, e.dist AS miles, e.rid AS rid)
) AS g
JOIN (VALUES ('BOS','Boston'), ('LAX','Los Angeles'), ('SEA','Seattle'), ('ORD','Chicago'))
    AS a(code, city) ON a.code = g.destination
ORDER BY g.miles DESC NULLS LAST, g.rid FETCH FIRST 2 ROWS ONLY;"""

with tempfile.TemporaryDirectory() as directory:
    graph = Path(directory) / "air_routes.lbdb"
    ladybug(lbug, graph, fixture.read_text(encoding="utf-8"))
    setup = f"LOAD {sql_literal(package)};\n" + provision(graph, alias="air_routes")
    expected = sorted([["ATL","BOS","946"], ["ATL","BOS","946"], ["ATL","LAX","1946"],
                       ["ATL","PDX","2172"], ["ATL","SEA","NULL"]])
    for lang in ("gql", "cypher"):
        select = f"SESSION SET GRAPH air_routes;\nSET SESSION qw_graph_dialect = '{lang}';\n"
        rows = read_rows(duckdb(cli, setup + select + graph_statement))
        assert sorted(rows) == expected, rows
        print(lang, sorted(rows))
    ranked = read_rows(duckdb(cli, setup + hybrid))
    assert ranked == [["Los Angeles", "1946"], ["Boston", "946"]], ranked
    print("SQL/PGQ", ranked)

gql [['ATL', 'BOS', '946'], ['ATL', 'BOS', '946'], ['ATL', 'LAX', '1946'], ['ATL', 'PDX', '2172'], ['ATL', 'SEA', 'NULL']]
cypher [['ATL', 'BOS', '946'], ['ATL', 'BOS', '946'], ['ATL', 'LAX', '1946'], ['ATL', 'PDX', '2172'], ['ATL', 'SEA', 'NULL']]


SQL/PGQ [['Los Angeles', '1946'], ['Boston', '946']]


## Appendix · Historical baseline for the `qw_query` bridge

The Rust cell below runs the packaged native extension at QuackWeave commit `d0bcebc` on DuckDB `v2.0.1` (source ID `8d78114e90`) and Ladybug `v0.22.3`. It creates two temporary Ladybug databases with parallel edges and different destination IDs. Each scenario checks exact returned rows, a plan property, or a diagnostic. The saved run passed 18 checks.

The cell uses the local Rust kernel and launches the pinned native CLIs. Run it from the QuackWeave workspace with the native release artifact built. It uses the binaries under `build/deps/duckdb-v2.0.1/duckdb` and `build/deps/ladybug-v0.22.3/lbug`. Set `QW_REPO`, `DUCKDB`, `LBUG`, or `QW_EXTENSION` when the corresponding local path differs.

### Air-routes fixture cross-check

At the initial design revision, 15 fixture and relational assertions passed on DuckDB v2.0.1 and Ladybug v0.22.3. The check substituted the current Cypher table function for the target graph table expression and ran the surrounding SQL. The five projected rows, empty result metadata, parallel-edge pair, directory joins (including Parquet), both limit placements, prepared filters, set operations and aggregates matched the examples. The result-reuse check materialized graph rows in DuckDB. Public GQL/Cypher entry points, native SQL/PGQ syntax, schema inference and driver registration require their own tests.

### Current bridge behavior verified by source and runtime

| Question | Pinned behavior |
| --- | --- |
| Which arguments can be prepared? | DuckDB resolves host values before binding the table function. The cell executes a prepared outer filter and a statement with placeholders for the whole payload and schema. The binder rejects a lateral row column. The shipped function has no `params :=` named argument for values inside the graph payload. |
| When is the graph identity fixed? | The first scan of an already-bound literal call checks the saved configuration generation and returns `QW_CONFIG` after an alias repoint. A fresh view, the tested prepared view, and a prepared call with a deferred payload each read the replacement graph when DuckDB binds the call again. The cell records `101, 101` before repointing and `201, 201` afterward. |
| Which composition belongs to DuckDB? | The same-alias GQL/Cypher join and graph-to-SQL join both preserved duplicates. Two graph rows joined to two SQL parent rows produced four results. `WITH ORDINALITY` numbered two delivered rows. The tested two-reference CTE had one graph scan operator when materialized and two when inlined. |
| When is the provider opened? | Plain `EXPLAIN` planned the GQL read against a corrupt fixture. `EXPLAIN ANALYZE` opened the fixture and returned `QW_OPEN`. |

The DuckDB `code_*` review used the registered `duckdb` project. The indexed symbols were checked against the runtime's pinned source revision. Table-function argument binding defers unresolved SQL parameters, and `EXECUTE` evaluates supplied values and rebinds when required. Column correlation has its own binder rule. [Argument binding](https://github.com/duckdb/duckdb/blob/8d78114e90a53823803660bf821f6073ac87da8c/src/planner/binder/tableref/bind_table_function.cpp#L97), [lateral argument binder](https://github.com/duckdb/duckdb/blob/8d78114e90a53823803660bf821f6073ac87da8c/src/planner/expression_binder/table_function_binder.cpp#L50), [prepare rebind](https://github.com/duckdb/duckdb/blob/8d78114e90a53823803660bf821f6073ac87da8c/src/main/prepared_statement_data.cpp#L96), [EXECUTE binder](https://github.com/duckdb/duckdb/blob/8d78114e90a53823803660bf821f6073ac87da8c/src/planner/binder/statement/bind_execute.cpp#L70)

DuckDB expands the saved view query through a child binder. QuackWeave snapshots the current alias configuration in its bind callback and checks the generation before the scan opens. The generation guard therefore protects the interval between a specific bind and that scan's first open. A guarantee that a named view stays on the same graph across rebinds would need an explicit graph identity checked at bind time. [DuckDB view expansion](https://github.com/duckdb/duckdb/blob/8d78114e90a53823803660bf821f6073ac87da8c/src/planner/binder/tableref/bind_basetableref.cpp#L308), [QuackWeave bind and scan](https://github.com/kevintruong/quack-weave/blob/d0bcebc558da8ac6a5f5aa7633a61cfcd0b15298/crates/core/src/qw_query.rs#L454)

CTE inlining and ordinality are host-plan operations. Use an outer `ORDER BY` on a stable application key for reproducible row ordering. The CTE evidence is a count of plan operators for the two tested forms. [DuckDB CTE inlining](https://github.com/duckdb/duckdb/blob/8d78114e90a53823803660bf821f6073ac87da8c/src/optimizer/cte_inlining.cpp#L165), [DuckDB ordinality](https://github.com/duckdb/duckdb/blob/8d78114e90a53823803660bf821f6073ac87da8c/src/planner/binder/tableref/bind_table_function.cpp#L363)

### JEV and Z3 receipts

JEV `1.14.0` opened all four gates. Z3 checked the saved cell evidence and the two bounded configuration traces.

| Check | Catalog rule | Verdict and receipt |
| --- | --- | --- |
| All 18 required checks appear in the saved Rust output | `data_integrity.foreign_key` | `sat` · `sol_33a266ff04b54ece` |
| Remove the deferred-payload identity check from the evidence | `data_integrity.foreign_key` | `unsat` · `sol_8a945899139b4d82` |
| Require the already-bound literal scan to avoid returning replacement rows | `workflow.safety_invariant` | `sat` · `sol_bc800cf5394c4eb2` |
| Apply the same requirement to the deferred-payload prepared statement | `workflow.safety_invariant` | `unsat` · `sol_4b3fb38c33014939` |

These receipts verify the historical bridge calls and host operations in the Rust cell. The coverage table above links the separate native GQL, Cypher, SQL/PGQ and ADBC tests.

In [4]:
{
    use std::{env, ffi::OsStr, fs, io::Write, path::{Path, PathBuf},
              process::{Command, Output, Stdio}, time::{SystemTime, UNIX_EPOCH}};

    fn invoke(bin: &Path, args: &[&OsStr], input: &str) -> Output {
        let mut child = Command::new(bin).args(args)
            .stdin(Stdio::piped()).stdout(Stdio::piped()).stderr(Stdio::piped())
            .spawn().unwrap_or_else(|e| panic!("Start {} failed: {e}", bin.display()));
        child.stdin.take().unwrap().write_all(input.as_bytes()).unwrap();
        child.wait_with_output().unwrap()
    }

    fn stdout(result: &Output) -> String {
        String::from_utf8(result.stdout.clone()).unwrap().trim().to_owned()
    }

    fn stderr(result: &Output) -> String {
        String::from_utf8_lossy(&result.stderr).into_owned()
    }

    fn expect(result: Output, label: &str, expected: &str) {
        assert!(result.status.success(), "{label}: {}", stderr(&result));
        assert_eq!(stdout(&result), expected, "{label}");
        println!("PASS {label}");
    }

    fn rejects(result: Output, label: &str, diagnostic: &str) {
        assert!(!result.status.success(), "{label}: unexpectedly returned {}", stdout(&result));
        assert!(stderr(&result).contains(diagnostic), "{label}: {}", stderr(&result));
        println!("PASS {label}");
    }

    fn sql_literal(s: &str) -> String {
        format!("'{}'", s.replace('\'', "''"))
    }

    fn sql_path(p: &Path) -> String {
        sql_literal(p.to_str().expect("UTF-8 path"))
    }

    fn provision(p: &Path) -> String {
        let p = p.to_str().expect("UTF-8 fixture path");
        assert!(!p.chars().any(char::is_control));
        let escaped = p.replace('\\', "\\\\").replace('"', "\\\"");
        let json = format!(r#"{{"alias":"kg","path":"{escaped}"}}"#);
        format!("SET GLOBAL qw_ladybug_config = {};\n", sql_literal(&json))
    }

    struct Scratch(PathBuf);
    impl Drop for Scratch {
        fn drop(&mut self) { let _ = fs::remove_dir_all(&self.0); }
    }

    let cwd = env::current_dir().unwrap();
    let root = env::var_os("QW_REPO").map(PathBuf::from).unwrap_or_else(|| {
        if cwd.join("test/stabilization/fixtures/read_graph.cypher").is_file() { cwd }
        else { PathBuf::from(env::var_os("HOME").expect("set QW_REPO")).join("project/quack-weave") }
    });
    let bin = |key: &str, path: &str| {
        env::var_os(key).map(PathBuf::from).unwrap_or_else(|| root.join(path))
    };
    let duckdb = bin("DUCKDB", "build/deps/duckdb-v2.0.1/duckdb");
    let lbug = bin("LBUG", "build/deps/ladybug-v0.22.3/lbug");
    let extension = bin("QW_EXTENSION", "build/native/release/extension/quack_weave/quack_weave.duckdb_extension");
    let version = invoke(&lbug, &[OsStr::new("-v")], "");
    expect(version, "runtime.ladybug", "Lbug 0.22.3");
    let query = |sql: &str| invoke(
        &duckdb,
        &["-no-agent", "-unsigned", "-batch", "-csv", "-noheader", "-bail", ":memory:"]
            .map(OsStr::new),
        sql,
    );
    expect(query("SELECT library_version, source_id FROM pragma_version();"),
           "runtime.duckdb", "v2.0.1,8d78114e90");

    let nonce = SystemTime::now().duration_since(UNIX_EPOCH).unwrap().as_nanos();
    let path = env::temp_dir().join(format!("qw-hybrid-{}-{nonce}", std::process::id()));
    fs::create_dir(&path).unwrap();
    let scratch = Scratch(path);
    let a = scratch.0.join("original.lbdb");
    let b = scratch.0.join("replacement.lbdb");
    let corrupt = scratch.0.join("corrupt.lbdb");
    fs::write(&corrupt, b"not a Ladybug graph").unwrap();
    for (file, key) in [(&a, 101), (&b, 201)] {
        let fixture = format!(r#"
CREATE NODE TABLE Document(id INT64 PRIMARY KEY);
CREATE REL TABLE LINKS_TO(FROM Document TO Document);
CREATE (:Document {{id: 100}}), (:Document {{id: {key}}});
MATCH (a:Document {{id: 100}}), (b:Document {{id: {key}}})
CREATE (a)-[:LINKS_TO]->(b), (a)-[:LINKS_TO]->(b);
"#);
        let seeded = invoke(&lbug,
            &[OsStr::new("-s"), OsStr::new("-b"), OsStr::new("-p"),
              scratch.0.as_os_str(), file.as_os_str()], &fixture);
        let diagnostics = format!("{}\n{}", stdout(&seeded), stderr(&seeded));
        assert!(seeded.status.success() && file.is_file()
            && !diagnostics.contains("Exception:") && !diagnostics.contains("Error:"),
            "Ladybug fixture setup failed: {diagnostics}");
    }

    let setup = format!("LOAD {};\n{}", sql_path(&extension), provision(&a));
    let exec = |sql: &str| query(&format!("{setup}{sql}"));
    let cypher = "MATCH (a:Document {id: 100})-[:LINKS_TO]->(b:Document) RETURN b.id AS document_id";
    let gql = "MATCH (a:Document)-[:LINKS_TO]->(b:Document) WHERE a.id = 100 RETURN b.id AS document_id";
    let schema = "schema := 'document_id BIGINT'";
    let c = format!("qw_query('kg', 'cypher', {}, {schema})", sql_literal(cypher));
    let g = format!("qw_query('kg', 'gql', {}, {schema})", sql_literal(gql));
    let select = format!("SELECT document_id FROM {c}");
    let replacement = provision(&b);

    expect(exec(&format!("SELECT document_id FROM {g} ORDER BY document_id;")),
           "relational.gql", "101\n101");
    expect(exec(&format!("SELECT count(*) FROM {c} c JOIN {g} g USING (document_id);")),
           "relational.same_alias_cross_dialect", "4");
    expect(exec(&format!("SELECT count(*) FROM {c} q JOIN (VALUES (101::BIGINT), (101::BIGINT)) p(id) ON q.document_id = p.id;")),
           "relational.duplicate_parents", "4");
    expect(exec(&format!("SELECT document_id, ordinality FROM {c} WITH ORDINALITY ORDER BY ordinality;")),
           "relational.ordinality", "101,1\n101,2");
    for (hint, count) in [("MATERIALIZED", 1), ("NOT MATERIALIZED", 2)] {
        let result = exec(&format!("EXPLAIN WITH r AS {hint} ({select}) SELECT count(*) FROM r a JOIN r b USING (document_id);"));
        assert!(result.status.success(), "{hint}: {}", stderr(&result));
        assert_eq!(stdout(&result).matches("QW_QUERY").count(), count, "{hint}");
    }
    println!("PASS relational.cte_plans");

    expect(exec(&format!("PREPARE host_filter AS SELECT count(*) FROM {c} WHERE document_id = $1; EXECUTE host_filter(101); EXECUTE host_filter(999);")),
           "binding.host_filter", "2\n0");
    expect(exec(&format!("PREPARE bound_args AS SELECT count(*) FROM qw_query('kg', 'cypher', $1, schema := $2); EXECUTE bound_args({}, 'document_id BIGINT');", sql_literal(cypher))),
           "binding.payload_and_schema", "2");
    rejects(exec("SELECT * FROM (VALUES ('MATCH (n:Document) RETURN n.id AS document_id')) t(q), LATERAL qw_query('kg','cypher',t.q,schema := 'document_id BIGINT');"),
            "binding.lateral_payload", "does not support lateral join column parameters");
    rejects(exec(&format!("SELECT * FROM qw_query('kg','cypher',{}, {schema}, params := '{{}}');", sql_literal(cypher))),
            "binding.no_graph_params_argument", "Invalid named parameter");
    expects_graph_macro(&exec, &c);

    expect(exec(&format!("CREATE VIEW linked AS {select}; SELECT * FROM linked ORDER BY 1; {replacement} SELECT * FROM linked ORDER BY 1;")),
           "identity.fresh_view_retargets", "101\n101\n201\n201");
    expect(exec(&format!("CREATE VIEW linked AS {select}; PREPARE saved AS SELECT * FROM linked; EXECUTE saved; {replacement} EXECUTE saved;")),
           "identity.prepared_view_retargets", "101\n101\n201\n201");
    let payload = sql_literal(cypher);
    expect(exec(&format!("PREPARE saved AS SELECT * FROM qw_query('kg', 'cypher', $1, {schema}); EXECUTE saved({payload}); {replacement} EXECUTE saved({payload});")),
           "identity.unresolved_payload_retargets", "101\n101\n201\n201");
    rejects(exec(&format!("PREPARE saved AS {select}; {replacement} EXECUTE saved;")),
            "identity.direct_prepared_rejects", "QW_CONFIG: Ladybug configuration changed after binding");

    let bad_path = provision(&corrupt);
    let plan = exec(&format!("{bad_path} EXPLAIN SELECT * FROM {g};"));
    assert!(plan.status.success() && stdout(&plan).contains("QW_QUERY"), "{}", stderr(&plan));
    println!("PASS execution.explain_without_open");
    rejects(exec(&format!("{bad_path} EXPLAIN ANALYZE SELECT * FROM {g};")),
            "execution.analyze_opens", "QW_OPEN");

    fn expects_graph_macro(exec: &impl Fn(&str) -> Output, c: &str) {
        expect(exec(&format!("CREATE MACRO count_linked(k) AS (SELECT count(*) FROM {c} WHERE document_id = k); SELECT count_linked(101), count_linked(999);")),
               "binding.host_macro", "2,0");
    }
}

PASS runtime.ladybug


PASS runtime.duckdb


PASS relational.gql


PASS relational.same_alias_cross_dialect


PASS relational.duplicate_parents


PASS relational.ordinality


... (9 outputs left out)

PASS identity.direct_prepared_rejects


PASS execution.explain_without_open


PASS execution.analyze_opens


()